In [83]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [84]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [85]:

queryCustomer = """
SELECT 
  [CustomerID],
  [TerritoryID],
  [PersonID],
  [AccountNumber]
FROM Sales.Customer
"""
tablaCustomer = pd.read_sql_query(queryCustomer, motorBaseDatos)



queryPersonPhone = """
SELECT 
  [BusinessEntityID],
  [PhoneNumber]
FROM Person.PersonPhone
"""
tablaPersonPhone = pd.read_sql_query(queryPersonPhone, motorBaseDatos)



queryPerson = """
SELECT 
  [BusinessEntityID],
  [NameStyle],
  [Title],
  [FirstName],
  [MiddleName],
  [LastName],
  [Suffix]
FROM Person.Person
"""
tablaPerson = pd.read_sql_query(queryPerson, motorBaseDatos)


queryEmployee = """
SELECT 
  [BusinessEntityID],
  [BirthDate],
  [MaritalStatus],
  [Gender]
FROM HumanResources.Employee
"""
tablaEmployee = pd.read_sql_query(queryEmployee, motorBaseDatos)



queryBusinessEntityAddress = """
SELECT 
  [BusinessEntityID],
  [AddressID]
FROM Person.BusinessEntityAddress
"""
tablaBusinessEntityAddress = pd.read_sql_query(queryBusinessEntityAddress, motorBaseDatos)



query = """
SELECT 
  [AddressID],
  [AddressLine1],
  [AddressLine2]
FROM Person.Address
"""
tablaAddress = pd.read_sql_query(query, motorBaseDatos)




queryEmailAddress = """
SELECT 
  [BusinessEntityID],
  [EmailAddress]
FROM Person.EmailAddress
"""

tablaEmailAddress = pd.read_sql_query(queryEmailAddress, motorBaseDatos)







TRANSFORMACION

In [86]:
address = tablaBusinessEntityAddress.merge(tablaAddress, on='AddressID')
address.drop('AddressID',axis=1, inplace=True)
address

,BusinessEntityID,AddressLine1,AddressLine2
0,1,4350 Minute Dr.,None
1,2,7559 Worth Ct.,None
2,3,2137 Birchwood Dr,None
3,4,5678 Lakeview Blvd.,None
4,5,9435 Breck Court,None
...,...,...,...
19609,20099,6097 Mt. McKinley Ct.,None
19610,20305,1960 Via Catanzaro,None
19611,20419,7723 Firestone Drive,None
19612,20550,7469 Paradise Ct.,None


In [87]:
dimensionCustomer = tablaPersonPhone.merge(tablaPerson, on='BusinessEntityID', how='left')
dimensionCustomer = dimensionCustomer.merge(address, on='BusinessEntityID', how='left')
dimensionCustomer = dimensionCustomer.merge(tablaEmployee, on='BusinessEntityID', how='left')
dimensionCustomer = dimensionCustomer.merge(tablaEmailAddress, on='BusinessEntityID', how='left')
dimensionCustomer = dimensionCustomer.merge(tablaCustomer, left_on='BusinessEntityID', right_on='PersonID', how='left')
dimensionCustomer.drop('PersonID', axis=1, inplace=True)


dimensionCustomer

,BusinessEntityID,PhoneNumber,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,BirthDate,MaritalStatus,Gender,EmailAddress,CustomerID,TerritoryID,AccountNumber
0,1959,1 (11) 500 555-0110,False,Mr.,Roger,None,Van Houten,None,NaN,NaN,NaN,NaN,NaN,roger2@adventure-works.com,30102.0,10.0,AW00030102
1,2409,1 (11) 500 555-0110,False,None,Laura,None,Zheng,None,4428 Jones Rd.,None,NaN,NaN,NaN,laura25@adventure-works.com,16701.0,9.0,AW00016701
2,2467,1 (11) 500 555-0110,False,None,Jamie,C,Hu,None,3022 Adobe St,None,NaN,NaN,NaN,jamie24@adventure-works.com,26772.0,9.0,AW00026772
3,2488,1 (11) 500 555-0110,False,None,Erica,R,Gao,None,1433 Manila Avenue,None,NaN,NaN,NaN,erica15@adventure-works.com,19087.0,9.0,AW00019087
4,2510,1 (11) 500 555-0110,False,None,Kristen,None,Xu,None,935 Vista Oak Dr,None,NaN,NaN,NaN,kristen11@adventure-works.com,15751.0,9.0,AW00015751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19991,14512,999-555-0149,False,None,Eduardo,L,Gray,None,3557 Harvard Court,None,NaN,NaN,NaN,eduardo74@adventure-works.com,18364.0,1.0,AW00018364
19992,13828,999-555-0152,False,None,Marcus,A,Powell,None,1707 Piper Ridge Court,None,NaN,NaN,NaN,marcus58@adventure-works.com,20366.0,4.0,AW00020366
19993,93,999-555-0155,False,None,Kok-Ho,T,Loh,None,3708 Montana,None,1980-04-28,S,M,kok-ho0@adventure-works.com,NaN,NaN,NaN
19994,6854,999-555-0183,False,None,Jordyn,None,Wood,None,33 RiverRock Dr,None,NaN,NaN,NaN,jordyn1@adventure-works.com,24521.0,1.0,AW00024521


In [88]:
# AGREGAR LAS COLUMNAS NUEVAS

dimensionCustomer["EnglishEducation"] = None
dimensionCustomer["SpanishEducation"] = None
dimensionCustomer["FrenchEducation"] = None
dimensionCustomer["EnglishOccupation"] = None
dimensionCustomer["SpanishOccupation"] = None
dimensionCustomer["FrenchOccupation"] = None
dimensionCustomer["YearlyIncome"] = None
dimensionCustomer["TotalChildren"] = None
dimensionCustomer["NumberChildrenAtHome"] = None
dimensionCustomer["HouseOwnerFlag"] = None
dimensionCustomer["NumberCarsOwned"] = None
dimensionCustomer["DateFirstPurchase"] = None
dimensionCustomer["CommuteDistance"] = None

dimensionCustomer

,BusinessEntityID,PhoneNumber,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,...,EnglishOccupation,SpanishOccupation,FrenchOccupation,YearlyIncome,TotalChildren,NumberChildrenAtHome,HouseOwnerFlag,NumberCarsOwned,DateFirstPurchase,CommuteDistance
0,1959,1 (11) 500 555-0110,False,Mr.,Roger,None,Van Houten,None,NaN,NaN,...,None,None,None,None,None,None,None,None,None,None
1,2409,1 (11) 500 555-0110,False,None,Laura,None,Zheng,None,4428 Jones Rd.,None,...,None,None,None,None,None,None,None,None,None,None
2,2467,1 (11) 500 555-0110,False,None,Jamie,C,Hu,None,3022 Adobe St,None,...,None,None,None,None,None,None,None,None,None,None
3,2488,1 (11) 500 555-0110,False,None,Erica,R,Gao,None,1433 Manila Avenue,None,...,None,None,None,None,None,None,None,None,None,None
4,2510,1 (11) 500 555-0110,False,None,Kristen,None,Xu,None,935 Vista Oak Dr,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19991,14512,999-555-0149,False,None,Eduardo,L,Gray,None,3557 Harvard Court,None,...,None,None,None,None,None,None,None,None,None,None
19992,13828,999-555-0152,False,None,Marcus,A,Powell,None,1707 Piper Ridge Court,None,...,None,None,None,None,None,None,None,None,None,None
19993,93,999-555-0155,False,None,Kok-Ho,T,Loh,None,3708 Montana,None,...,None,None,None,None,None,None,None,None,None,None
19994,6854,999-555-0183,False,None,Jordyn,None,Wood,None,33 RiverRock Dr,None,...,None,None,None,None,None,None,None,None,None,None


In [89]:
dimensionCustomer.rename(columns={
    "CustomerID":"CustomerKey",
    "AccountNumber":"CustomerAlternateKey",
}, inplace=True)

dimensionCustomer

,BusinessEntityID,PhoneNumber,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,...,EnglishOccupation,SpanishOccupation,FrenchOccupation,YearlyIncome,TotalChildren,NumberChildrenAtHome,HouseOwnerFlag,NumberCarsOwned,DateFirstPurchase,CommuteDistance
0,1959,1 (11) 500 555-0110,False,Mr.,Roger,None,Van Houten,None,NaN,NaN,...,None,None,None,None,None,None,None,None,None,None
1,2409,1 (11) 500 555-0110,False,None,Laura,None,Zheng,None,4428 Jones Rd.,None,...,None,None,None,None,None,None,None,None,None,None
2,2467,1 (11) 500 555-0110,False,None,Jamie,C,Hu,None,3022 Adobe St,None,...,None,None,None,None,None,None,None,None,None,None
3,2488,1 (11) 500 555-0110,False,None,Erica,R,Gao,None,1433 Manila Avenue,None,...,None,None,None,None,None,None,None,None,None,None
4,2510,1 (11) 500 555-0110,False,None,Kristen,None,Xu,None,935 Vista Oak Dr,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19991,14512,999-555-0149,False,None,Eduardo,L,Gray,None,3557 Harvard Court,None,...,None,None,None,None,None,None,None,None,None,None
19992,13828,999-555-0152,False,None,Marcus,A,Powell,None,1707 Piper Ridge Court,None,...,None,None,None,None,None,None,None,None,None,None
19993,93,999-555-0155,False,None,Kok-Ho,T,Loh,None,3708 Montana,None,...,None,None,None,None,None,None,None,None,None,None
19994,6854,999-555-0183,False,None,Jordyn,None,Wood,None,33 RiverRock Dr,None,...,None,None,None,None,None,None,None,None,None,None


In [90]:
dimensionGeography = pd.read_sql_table("dimensionGeography",motorBodegaDatos)
dimensionGeography.drop(columns=[
    'City',
      'CountryRegionCode',
      'PostalCode',
      'StateProvinceCode',
      'StateProvinceName',
      'SalesTerritoryKey',
      'EnglishCountryRegionName',
      'SpanishCountryRegionName',
      'FrenchCountryRegionName',
      'IpAddressLocator'
], inplace=True)
dimensionCustomer = dimensionCustomer.merge(dimensionGeography, left_on='TerritoryID', right_on='GeographyKey',how='left')


dimensionCustomer.columns

Index(['BusinessEntityID', 'PhoneNumber', 'NameStyle', 'Title', 'FirstName',
       'MiddleName', 'LastName', 'Suffix', 'AddressLine1', 'AddressLine2',
       'BirthDate', 'MaritalStatus', 'Gender', 'EmailAddress', 'CustomerKey',
       'TerritoryID', 'CustomerAlternateKey', 'EnglishEducation',
       'SpanishEducation', 'FrenchEducation', 'EnglishOccupation',
       'SpanishOccupation', 'FrenchOccupation', 'YearlyIncome',
       'TotalChildren', 'NumberChildrenAtHome', 'HouseOwnerFlag',
       'NumberCarsOwned', 'DateFirstPurchase', 'CommuteDistance',
       'GeographyKey'],
      dtype='object')

CARGAR A LA BODEGA

In [93]:
dimensionCustomer.to_sql('dimensionCustomer',motorBodegaDatos, if_exists='replace',index=False)

30